# EmpowerLens — D25 app models (Kaggle GPU)

Retrains the **D12 winner** for the demo app and keeps the weights (D09/D13 weights were not kept):

1. `binary_mental-roberta-base_42` — MentalRoBERTa binary, seed 42, T14 settings; D12 failed-run rule (val macro-F1
   below 0.5 → re-run once); then the **D13 threshold** (`src.fit_binary_threshold --target-fpr 0.15`, fitted on val).
2. `multilabel_mental-roberta-base_42` — same settings, multilabel; per-type thresholds fitted on val during training.
3. `span_selector/model.pt` — the **D03 sentence selector**, seed 42 (the app uses it only to pick example sentences).

Both classifiers are then scored once by `src.evaluate` (the only script that reads test). Everything the app needs,
plus `MODEL_CARD.json` (name, version, thresholds, label order, scores), goes to `/kaggle/working/app_models/` and is
zipped as `app_models.zip` (~1.5 GB). About 60–80 min on a T4.

**Before running:** Accelerator **GPU (T4 or P100)**, **Internet: On**, Kaggle secret **`HF_TOKEN`**
(`mental/mental-roberta-base` is gated).

**Resumable:** each step is skipped if its output is already in `/kaggle/working/app_models/`.
**After it finishes:** download `app_models.zip` and unzip it so the demo repo has `models/d12/MODEL_CARD.json`
(i.e. unzip into `empowerlens-demo/models/` and rename `app_models` to `d12`). Weights never go into git.

In [ ]:
# 1. HF token, single GPU, clone the repo into /kaggle/temp (not saved with the output), install.
import os
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    print(f"[warn] no HF_TOKEN secret ({e}) - mental-roberta-base will fail to download.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # avoid multi-GPU load hangs on T4 x2

REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"
!rm -rf /kaggle/temp/empowerlens && mkdir -p /kaggle/temp && git clone --depth 1 --branch $BRANCH $REPO_URL /kaggle/temp/empowerlens
%cd /kaggle/temp/empowerlens
!pip install -q -r requirements-transformer.txt
!ls src/fit_binary_threshold.py src/train_span_selector.py results/results_sentence_label/sentence_labels_train.csv && python -c "import torch; print('cuda:', torch.cuda.is_available())"

In [ ]:
# 2. Shared config + helpers (re-run this cell first after any kernel restart).
import os, json, shutil, subprocess, time, datetime
from pathlib import Path
os.chdir("/kaggle/temp/empowerlens")

MODEL  = "mental/mental-roberta-base"
TAG    = MODEL.split("/")[-1]
SEED   = 42
OUT    = Path("/kaggle/working/app_models")
CK     = Path("/kaggle/temp/ckpt_d25")
EVAL   = OUT / "eval"
RECIPE = (f"--model {MODEL} --seed {SEED} --lr 3e-5 --epochs 5 --weight-decay 0.1 "
          f"--max-length 512 --batch-size 16 --splits data/splits")
OUT.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    print(f"\n$ {cmd}", flush=True)
    t = time.time()
    subprocess.run(cmd, shell=True, check=True)
    print(f"[done in {(time.time() - t) / 60:.1f} min]", flush=True)

def meta(ck):
    return json.loads((Path(ck) / "meta.json").read_text())

def keep(ck):
    """Copy a trained checkpoint to OUT without the Trainer's checkpoint-* folders (optimizer state)."""
    dst = OUT / Path(ck).name
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(ck, dst, ignore=shutil.ignore_patterns("checkpoint-*"))
    return dst

print("config ok:", MODEL, SEED, OUT)

In [ ]:
# 3. Smoke test (~3 min): tiny binary run + tiny selector run. Writes nothing to OUT.
sh(f"python -m src.train_transformer --task binary --model {MODEL} --seed 42 --smoke --max-length 128 "
   f"--out /kaggle/temp/smoke")
sh("python -m src.train_span_selector --seed 42 --smoke --ckpt-dir /kaggle/temp/smoke_sel")
assert Path("/kaggle/temp/smoke_sel/42/model.pt").exists()
print("SMOKE OK")
shutil.rmtree("/kaggle/temp/smoke", ignore_errors=True); shutil.rmtree("/kaggle/temp/smoke_sel", ignore_errors=True)

In [ ]:
# 4. Binary classifier + D13 threshold (FPR <= 0.15 on val) + one test evaluation.
b_name = f"binary_{TAG}_{SEED}"
if not (OUT / b_name / "meta.json").exists() or "binary_threshold" not in meta(OUT / b_name):
    reruns = 0
    sh(f"python -m src.train_transformer --task binary {RECIPE} --out {CK}")
    f1 = meta(CK / b_name)["val_metrics"]["eval_macro_f1"]
    if f1 < 0.5:   # D12 failed-run rule: re-run once with identical settings
        print(f"[failed-run rule] val macro-F1 {f1:.3f} < 0.5 -> re-running once")
        reruns = 1
        sh(f"python -m src.train_transformer --task binary {RECIPE} --out {CK}")
    sh(f"python -m src.fit_binary_threshold --checkpoint {CK / b_name} --target-fpr 0.15")
    m = meta(CK / b_name); m["failed_run_reruns"] = reruns
    (CK / b_name / "meta.json").write_text(json.dumps(m, indent=2))
    keep(CK / b_name)
sh(f"python -m src.evaluate --checkpoint {OUT / b_name} --out {EVAL}")
m = meta(OUT / b_name)
print("binary val macro-F1", round(m["val_metrics"]["eval_macro_f1"], 3), "threshold", m["binary_threshold"])

In [ ]:
# 5. Multilabel classifier (per-type thresholds fitted on val during training) + one test evaluation.
m_name = f"multilabel_{TAG}_{SEED}"
if not (OUT / m_name / "meta.json").exists():
    sh(f"python -m src.train_transformer --task multilabel {RECIPE} --out {CK}")
    keep(CK / m_name)
sh(f"python -m src.evaluate --checkpoint {OUT / m_name} --out {EVAL}")
m = meta(OUT / m_name)
print("multilabel val macro-F1", round(m["val_metrics"]["eval_macro_f1"], 3), "thresholds", m["thresholds"])

In [ ]:
# 6. D03 sentence selector, seed 42 (threshold chosen on val inside the script).
SEL = OUT / "span_selector"
if not (SEL / "model.pt").exists():
    sh(f"python -m src.train_span_selector --seed {SEED} --model {MODEL} --ckpt-dir {CK}/span_selector")
    SEL.mkdir(parents=True, exist_ok=True)
    shutil.copy(CK / "span_selector" / str(SEED) / "model.pt", SEL / "model.pt")
    for part in ("val", "test"):
        shutil.copy(f"results/results_span_selector/eval_{part}_{SEED}.json", SEL / f"eval_{part}_{SEED}.json")
sel = json.loads((SEL / f"eval_val_{SEED}.json").read_text())
print("selector val sentence F1", round(sel["f1"], 3), "threshold", sel["threshold"])

In [ ]:
# 7. MODEL_CARD.json (what the app reads) + zip for download.
from src.data import DISTORTIONS

def split_metrics(task):
    e = json.loads(next(EVAL.glob(f"eval_*_{task}_{SEED}.json")).read_text())["splits"]
    return {s: e[s]["metrics"] for s in ("val", "test") if s in e}

bm, mm = meta(OUT / b_name), meta(OUT / m_name)
sel_val = json.loads((SEL / f"eval_val_{SEED}.json").read_text())
sel_test = json.loads((SEL / f"eval_test_{SEED}.json").read_text())
today = datetime.date.today()
card = {
    "model_name": "MentalRoBERTa",
    "model_id": MODEL,
    "model_version": f"mentalroberta-d12-s{SEED}-{today:%Y%m%d}",
    "trained_at": today.isoformat(),
    "main_repo_commit": subprocess.check_output("git rev-parse --short HEAD", shell=True, text=True).strip(),
    "selected_by": "docs/model_selection.md (D12): highest val multilabel macro-F1 at the D13 operating point",
    "threshold_rule": "docs/thresholds.md (D13): binary = lowest val threshold with FPR <= 0.15; "
                      "multilabel = per-type threshold with best val F1",
    "seed": SEED,
    "settings": {k: bm[k] for k in ("lr", "epochs", "weight_decay", "batch_size", "max_length", "truncation")},
    "encoding": {"max_length": bm["max_length"], "truncation": bm["truncation"],
                 "special_tokens": "added manually around the text, as in src/train_transformer.encode_texts"},
    "binary": {"checkpoint": b_name, "threshold": bm["binary_threshold"],
               "threshold_criterion": bm.get("binary_threshold_criterion"),
               "failed_run_reruns": bm.get("failed_run_reruns", 0), "metrics": split_metrics("binary")},
    "multilabel": {"checkpoint": m_name, "labels": list(DISTORTIONS), "thresholds": mm["thresholds"],
                   "max_labels": 2, "metrics": split_metrics("multilabel")},
    "selector": {"checkpoint": "span_selector/model.pt", "base_model": MODEL, "max_length": 128,
                 "threshold": sel_val["threshold"], "input": "one sentence, no context",
                 "sentence_splitter": "src/span_check.py split_sentences",
                 "val": {k: sel_val[k] for k in ("precision", "recall", "f1", "row_hit_rate")},
                 "test": {k: sel_test[k] for k in ("precision", "recall", "f1", "row_hit_rate")}},
}
(OUT / "MODEL_CARD.json").write_text(json.dumps(card, indent=2))
print(json.dumps({k: card[k] for k in ("model_version", "main_repo_commit")}, indent=2))
print("binary threshold", card["binary"]["threshold"], "| selector threshold", card["selector"]["threshold"])
!cd /kaggle/working && rm -f app_models.zip && zip -qr app_models.zip app_models
!du -sh /kaggle/working/app_models /kaggle/working/app_models.zip